In [ ]:
import pandas as pd
from pathlib import Path

STORE = Path('/kaggle/input/notebooks/waithakafrancis/data-pipeline-v1/feature_store')

a_train = pd.read_parquet(STORE / 'pipeline_a_train.parquet')
b_train = pd.read_parquet(STORE / 'pipeline_b_train.parquet')
holdout = pd.read_parquet(STORE / 'holdout.parquet')

print('R-GAN training shape    :', a_train.shape)
print('Pipeline B training     :', b_train.shape)
print('Holdout shape           :', holdout.shape)

In [2]:
# ── Verify your data loaded correctly ────────────────────────────
# R-GAN: should see ~9% fraud rate (post-SMOTE)
fraud_rate_a = a_train['Class'].mean()
print(f'Pipeline A fraud rate: {fraud_rate_a:.2%}')   # expect ~9%

# f-AnoGAN: must have 0% fraud
fraud_rate_b = b_train['Class'].mean()
print(f'Pipeline B fraud rate: {fraud_rate_b:.2%}')   # expect 0.00%

# Feature count: should be 51
print(f'Features: {a_train.shape[1]}')               # expect 51

# Separate features and labels for R-GAN
X_rgan = a_train.drop(columns=['Class'])
y_rgan = a_train['Class']
print(f'X shape: {X_rgan.shape}, fraud samples: {y_rgan.sum():,}')


Pipeline A fraud rate: 9.09%
Pipeline B fraud rate: 0.00%
Features: 51
X shape: (250170, 50), fraud samples: 22,742


In [ ]:
STORE = Path('/kaggle/input/notebooks/waithakafrancis/data-pipeline-v1/feature_store')

a_train = pd.read_parquet(STORE / 'pipeline_a_train.parquet')
a_val   = pd.read_parquet(STORE / 'pipeline_a_val.parquet')
holdout = pd.read_parquet(STORE / 'holdout.parquet')

assert list(a_train.columns) == list(a_val.columns) == list(holdout.columns), \
    "Column mismatch between train/val/holdout"

print(f'Pipeline A train shape : {a_train.shape}')
print(f'Pipeline A val shape   : {a_val.shape}')
print(f'Holdout shape          : {holdout.shape}')
print(f'Pipeline A fraud rate  : {a_train["Class"].mean():.4%}')
print(f'Feature count          : {a_train.shape[1]}')

In [ ]:
import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import wandb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, classification_report,
    matthews_corrcoef,
)
from imblearn.metrics import geometric_mean_score
from scipy.spatial.distance import jensenshannon
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GLOBAL_SEED = 42

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(GLOBAL_SEED)
print(f'Device: {DEVICE}')
print('Deterministic mode enabled.')

In [8]:
wandb.login(key='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')


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
EXPECTED_COLS   = 31   # Time + V1-V28 + Amount + Class
EXPECTED_TARGET = 'Class'

# Time is excluded from model features per FIX 8 (Hayat & Magnier)
MODEL_FEATURES = [f'V{i}' for i in range(1, 29)] + ['Amount']  # 29 features
GAN_FEATURES   = MODEL_FEATURES  # GAN trains on the same 29 features

def validate_schema(df, name, expected_fraud_rate=None, tolerance=0.05):
    errors = []
    if df.shape[1] != EXPECTED_COLS:
        errors.append(f'expected {EXPECTED_COLS} cols, got {df.shape[1]}')
    if EXPECTED_TARGET not in df.columns:
        errors.append(f'missing target column "{EXPECTED_TARGET}"')
    if df.isnull().sum().sum() > 0:
        errors.append(f'contains {df.isnull().sum().sum()} null values')
    if not set(df[EXPECTED_TARGET].unique()).issubset({0, 1}):
        errors.append(f'Class contains non-binary values')
    if expected_fraud_rate is not None:
        actual = df[EXPECTED_TARGET].mean()
        if abs(actual - expected_fraud_rate) > tolerance:
            errors.append(
                f'fraud rate {actual:.4%} deviates from expected {expected_fraud_rate:.4%}'
            )
    dups = df.duplicated().sum()
    if dups > 0:
        print(f'  WARNING: {name} has {dups} duplicate rows')
    if errors:
        for e in errors:
            print(f'  SCHEMA ERROR [{name}]: {e}')
        raise ValueError(f'Schema validation failed for {name}.')
    print(
        f'  {name}: OK — shape={df.shape}, '
        f'fraud_rate={df[EXPECTED_TARGET].mean():.4%}, nulls=0'
    )

print('Running schema validation...')
validate_schema(a_train, 'a_train', expected_fraud_rate=0.002, tolerance=0.005)
validate_schema(a_val,   'a_val',   expected_fraud_rate=None)
validate_schema(holdout, 'holdout', expected_fraud_rate=None)
print('All schema checks passed.')

X_rgan = a_train[MODEL_FEATURES]
y_rgan = a_train[EXPECTED_TARGET]

print(f'\nX shape      : {X_rgan.shape}')
print(f'Fraud samples: {y_rgan.sum():,}  ({y_rgan.mean():.4%})')

In [10]:
@dataclass
class RGANConfig:
    # ── Data ──────────────────────────────────────────────────────────────
    target_col: str = 'Class'
    n_features: int = 50        # set dynamically below

    # ── Architecture ──────────────────────────────────────────────────────
    latent_dim:  int = 128
    gen_dim_1:   int = 256
    gen_dim_2:   int = 512
    gen_dim_3:   int = 256
    disc_dim_1:  int = 512
    disc_dim_2:  int = 256
    disc_dim_3:  int = 128

    # ── GAN Training ──────────────────────────────────────────────────────
    batch_size:     int   = 256
    n_epochs:       int   = 300
    lr_g:           float = 1e-4
    lr_d:           float = 1e-4
    adam_beta1:     float = 0.5
    adam_beta2:     float = 0.9
    gp_lambda:      float = 10.0
    n_critic:       int   = 5

    # ── Similarity Measure Loss ────────────────────────────────────────────
    # Weight for the statistical similarity penalty between
    # synthetic fraud and real fraud (mean + std matching)
    lambda_sim:     float = 0.5

    # ── Early stopping ────────────────────────────────────────────────────
    patience:       int   = 25

    # ── LightGBM ──────────────────────────────────────────────────────────
    lgb_num_leaves:          int   = 63
    lgb_learning_rate:       float = 0.05
    lgb_n_estimators:        int   = 500
    lgb_min_child_samples:   int   = 20
    lgb_subsample:           float = 0.8
    lgb_colsample_bytree:    float = 0.8
    lgb_reg_alpha:           float = 0.1
    lgb_reg_lambda:          float = 0.1
    lgb_early_stopping_rounds: int = 50

    # ── Synthetic generation ──────────────────────────────────────────────
    n_synthetic_fraud: int = 5000

    # ── Quality gate thresholds ───────────────────────────────────────────
    mmd_threshold:          float = 0.05
    jsd_threshold:          float = 0.10
    novelty_min_distance:   float = 0.01
    disc_auc_threshold:     float = 0.75


cfg            = RGANConfig()
cfg.n_features = X_rgan.shape[1]   # 50

print(
    f'Config: n_features={cfg.n_features}, latent_dim={cfg.latent_dim}, '
    f'n_epochs={cfg.n_epochs}, lambda_sim={cfg.lambda_sim}'
)

Config: n_features=50, latent_dim=128, n_epochs=300, lambda_sim=0.5


In [11]:
run = wandb.init(
    project = 'PRINCIPLES AND ENGINEERING APPLICATIONS OF AI',
    name    = 'rgan-pipeline-a-run-1',
    tags    = ['R-GAN', 'Pipeline-A', 'WGAN-GP', 'LightGBM'],
    config  = asdict(cfg),
)

wcfg                = wandb.config
cfg.n_features      = wcfg.n_features
cfg.latent_dim      = wcfg.latent_dim
cfg.batch_size      = wcfg.batch_size
cfg.n_epochs        = wcfg.n_epochs
cfg.lr_g            = wcfg.lr_g
cfg.lr_d            = wcfg.lr_d
cfg.gp_lambda       = wcfg.gp_lambda
cfg.n_critic        = wcfg.n_critic
cfg.lambda_sim      = wcfg.lambda_sim
cfg.n_synthetic_fraud        = wcfg.n_synthetic_fraud
cfg.lgb_num_leaves           = wcfg.lgb_num_leaves
cfg.lgb_learning_rate        = wcfg.lgb_learning_rate
cfg.lgb_n_estimators         = wcfg.lgb_n_estimators
cfg.lgb_subsample            = wcfg.lgb_subsample
cfg.lgb_colsample_bytree     = wcfg.lgb_colsample_bytree
cfg.lgb_reg_alpha            = wcfg.lgb_reg_alpha
cfg.lgb_reg_lambda           = wcfg.lgb_reg_lambda

print('W&B run initialised:', run.name)
print('W&B project URL    :', run.url)

W&B run initialised: rgan-pipeline-a-run-1
W&B project URL    : https://wandb.ai/kkipngenokoech-carnegie-mellon-university/PRINCIPLES%20AND%20ENGINEERING%20APPLICATIONS%20OF%20AI/runs/9sltcyqj


In [ ]:
# GAN_FEATURES already defined in cell-5 as MODEL_FEATURES (V1-V28 + Amount)
fraud_mask       = y_rgan == 1
fraud_train_gan  = X_rgan.loc[fraud_mask, GAN_FEATURES].reset_index(drop=True)

N_REAL_FRAUD          = len(fraud_train_gan)
cfg.n_synthetic_fraud = int(N_REAL_FRAUD * 3)   # 3x expansion

print(f'Fraud records for R-GAN training : {N_REAL_FRAUD}')
print(f'GAN training features            : {len(GAN_FEATURES)}')
print(f'Synthetic budget (3x)            : {cfg.n_synthetic_fraud}')

rgan_scaler        = StandardScaler()
fraud_train_scaled = np.clip(
    rgan_scaler.fit_transform(fraud_train_gan.values.astype(np.float32)),
    -5, 5
).astype(np.float32)

print(f'Scaled range: {fraud_train_scaled.min():.3f} to {fraud_train_scaled.max():.3f}')

real_fraud_tensor = torch.tensor(fraud_train_scaled, dtype=torch.float32)
cfg.n_features    = len(GAN_FEATURES)  # 29

fraud_dataset = TensorDataset(real_fraud_tensor)
fraud_loader  = DataLoader(
    fraud_dataset,
    batch_size = cfg.batch_size,
    shuffle    = True,
    drop_last  = True,
)

print(f'cfg.n_features updated to : {cfg.n_features}')
print(f'Batches per epoch         : {len(fraud_loader)}')

wandb.log({
    'data/n_fraud_train'    : N_REAL_FRAUD,
    'data/n_gan_features'   : len(GAN_FEATURES),
    'data/batches_per_epoch': len(fraud_loader),
})

In [13]:
def spectral_norm(module):
    """Apply spectral normalisation — key stabiliser in Advanced R-GAN."""
    return nn.utils.spectral_norm(module)


class RGANGenerator(nn.Module):
    """
    R-GAN Generator with:
    - Spectral normalisation on all linear layers
    - CELU activations (smoother gradient flow than ReLU)
    - No final activation (features are unbounded)
    """
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Linear(cfg.latent_dim, cfg.gen_dim_1)),
            nn.BatchNorm1d(cfg.gen_dim_1),
            nn.CELU(inplace=True),

            spectral_norm(nn.Linear(cfg.gen_dim_1, cfg.gen_dim_2)),
            nn.BatchNorm1d(cfg.gen_dim_2),
            nn.CELU(inplace=True),

            spectral_norm(nn.Linear(cfg.gen_dim_2, cfg.gen_dim_3)),
            nn.BatchNorm1d(cfg.gen_dim_3),
            nn.CELU(inplace=True),

            spectral_norm(nn.Linear(cfg.gen_dim_3, cfg.n_features)),
            # No Tanh — real features are not bounded to [-1, 1]
        )

    def forward(self, z):
        return self.net(z)


class RGANDiscriminator(nn.Module):
    """
    R-GAN Discriminator (WGAN-GP critic) with:
    - Spectral normalisation
    - LeakyReLU activations
    - No BatchNorm — required for WGAN-GP gradient penalty
    Returns (score, intermediate_features) — features used in
    Similarity Measure Loss feature matching component.
    """
    def __init__(self, cfg):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            spectral_norm(nn.Linear(cfg.n_features, cfg.disc_dim_1)),
            nn.LeakyReLU(0.2, inplace=True),

            spectral_norm(nn.Linear(cfg.disc_dim_1, cfg.disc_dim_2)),
            nn.LeakyReLU(0.2, inplace=True),

            spectral_norm(nn.Linear(cfg.disc_dim_2, cfg.disc_dim_3)),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.critic = spectral_norm(nn.Linear(cfg.disc_dim_3, 1))

    def forward(self, x):
        features = self.feature_extractor(x)
        score    = self.critic(features)
        return score, features


# ── Instantiate ───────────────────────────────────────────────────────────
G = RGANGenerator(cfg).to(DEVICE)
D = RGANDiscriminator(cfg).to(DEVICE)

n_g = sum(p.numel() for p in G.parameters())
n_d = sum(p.numel() for p in D.parameters())

print(f'Generator params    : {n_g:,}')
print(f'Discriminator params: {n_d:,}')
print(f'Total params        : {n_g + n_d:,}')

# ── Xavier weight init ────────────────────────────────────────────────────
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

G.apply(weights_init)
D.apply(weights_init)
print('Weights initialised.')

# ── Smoke test ────────────────────────────────────────────────────────────
_z    = torch.randn(4, cfg.latent_dim).to(DEVICE)
_x    = torch.randn(4, cfg.n_features).to(DEVICE)
_fake = G(_z)
_score, _feat = D(_x)
print(
    f'Smoke test OK — '
    f'z: {_z.shape} | G(z): {_fake.shape} | '
    f'D score: {_score.shape} | D feat: {_feat.shape}'
)

wandb.log({
    'model/generator_params'    : n_g,
    'model/discriminator_params': n_d,
    'model/total_params'        : n_g + n_d,
})

Generator params    : 305,437
Discriminator params: 179,713
Total params        : 485,150
Weights initialised.
Smoke test OK — z: torch.Size([4, 128]) | G(z): torch.Size([4, 29]) | D score: torch.Size([4, 1]) | D feat: torch.Size([4, 128])


In [14]:
def similarity_measure_loss(fake, real_batch, D):
    """
    R-GAN Similarity Measure Loss — the key differentiator from standard WGAN.

    Two components:
    1. Statistical similarity: penalises difference in per-feature
       mean and std between synthetic and real fraud batches.
       This constrains the generator to preserve the statistical
       signatures of genuine fraud records.

    2. Feature matching: penalises difference in discriminator
       intermediate features between synthetic and real batches.
       This encourages the generator to produce samples that are
       indistinguishable from real fraud in the learned feature space.

    Both are computed against the REAL FRAUD batch passed in,
    not against random noise — this is why X and y must be
    separated before training.
    """
    # ── Component 1: Statistical similarity ──────────────────────────────
    fake_mean = fake.mean(dim=0)
    fake_std  = fake.std(dim=0)  + 1e-8
    real_mean = real_batch.mean(dim=0)
    real_std  = real_batch.std(dim=0)  + 1e-8

    loss_mean = torch.mean((fake_mean - real_mean) ** 2)
    loss_std  = torch.mean((fake_std  - real_std)  ** 2)
    stat_loss = loss_mean + loss_std

    # ── Component 2: Feature matching ─────────────────────────────────────
    with torch.no_grad():
        _, feat_real = D(real_batch)
    _, feat_fake = D(fake)
    feat_loss = torch.mean((feat_fake - feat_real.detach()) ** 2)

    return stat_loss + feat_loss, stat_loss.item(), feat_loss.item()

In [15]:
def compute_gradient_penalty(D, real, fake, device):
    bs     = real.size(0)
    alpha  = torch.rand(bs, 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp, _ = D(interp)
    grads  = torch.autograd.grad(
        outputs      = d_interp,
        inputs       = interp,
        grad_outputs = torch.ones_like(d_interp),
        create_graph = True,
        retain_graph = True,
    )[0]
    return ((grads.view(bs, -1).norm(2, dim=1) - 1) ** 2).mean()


def train_rgan(G, D, loader, cfg, device,
               checkpoint_path='/kaggle/working/rgan_best.pt'):
    """
    R-GAN WGAN-GP training loop with:
    - Distinct real batches per critic step
    - Similarity Measure Loss on generator update
    - Best model checkpointing
    - Early stopping on Wasserstein distance
    """
    opt_G = optim.Adam(G.parameters(), lr=cfg.lr_g,
                       betas=(cfg.adam_beta1, cfg.adam_beta2))
    opt_D = optim.Adam(D.parameters(), lr=cfg.lr_d,
                       betas=(cfg.adam_beta1, cfg.adam_beta2))

    history = {
        'g_loss': [], 'd_loss': [], 'gp': [],
        'sim_loss': [], 'wasserstein': []
    }

    best_w_dist   = float('inf')
    patience_ctr  = 0
    best_state_G  = None
    best_state_D  = None

    for epoch in range(cfg.n_epochs):
        G.train(); D.train()
        g_losses, d_losses = [], []
        gp_vals, sim_losses = [], []

        # ── Iterator for distinct critic batches ──────────────────────────
        data_iter = iter(loader)

        def next_batch():
            nonlocal data_iter
            try:
                return next(data_iter)[0].to(device)
            except StopIteration:
                data_iter = iter(loader)
                return next(data_iter)[0].to(device)

        for _ in range(len(loader)):

            # ── Critic: n_critic steps, each with a distinct real batch ───
            for _ in range(cfg.n_critic):
                x_real = next_batch()
                bs     = x_real.size(0)

                z    = torch.randn(bs, cfg.latent_dim, device=device)
                fake = G(z).detach()

                gp      = compute_gradient_penalty(D, x_real, fake, device)
                d_real, _ = D(x_real)
                d_fake, _ = D(fake)
                d_loss    = (
                    d_fake.mean() - d_real.mean()
                    + cfg.gp_lambda * gp
                )
                opt_D.zero_grad()
                d_loss.backward()
                torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
                opt_D.step()

                d_losses.append(d_loss.item())
                gp_vals.append(gp.item())

            # ── Generator step ────────────────────────────────────────────
            x_real = next_batch()
            bs     = x_real.size(0)

            z    = torch.randn(bs, cfg.latent_dim, device=device)
            fake = G(z)

            # Standard WGAN generator loss
            g_score, _ = D(fake)
            g_loss_wgan = -g_score.mean()

            # Similarity Measure Loss — requires real fraud batch
            sim_loss, stat_l, feat_l = similarity_measure_loss(
                fake, x_real, D
            )

            g_loss = g_loss_wgan + cfg.lambda_sim * sim_loss

            opt_G.zero_grad()
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
            opt_G.step()

            g_losses.append(g_loss.item())
            sim_losses.append(sim_loss.item())

        mg    = float(np.mean(g_losses))
        md    = float(np.mean(d_losses))
        mgp   = float(np.mean(gp_vals))
        msim  = float(np.mean(sim_losses))
        w_dist = -md

        history['g_loss'].append(mg)
        history['d_loss'].append(md)
        history['gp'].append(mgp)
        history['sim_loss'].append(msim)
        history['wasserstein'].append(w_dist)

        # ── Checkpoint best model ─────────────────────────────────────────
        if w_dist < best_w_dist:
            best_w_dist  = w_dist
            best_state_G = deepcopy(G.state_dict())
            best_state_D = deepcopy(D.state_dict())
            torch.save(
                {'G': best_state_G, 'D': best_state_D,
                 'epoch': epoch + 1, 'w_dist': best_w_dist},
                checkpoint_path
            )
            patience_ctr = 0
        else:
            patience_ctr += 1

        # ── W&B logging ───────────────────────────────────────────────────
        wandb.log({
            'train/epoch'              : epoch + 1,
            'train/g_loss'             : mg,
            'train/d_loss'             : md,
            'train/gradient_penalty'   : mgp,
            'train/similarity_loss'    : msim,
            'train/wasserstein_distance': w_dist,
            'train/best_w_dist'        : best_w_dist,
            'train/patience'           : patience_ctr,
        }, step=epoch + 1)

        if (epoch + 1) % 10 == 0:
            print(
                f'Epoch [{epoch+1:3d}/{cfg.n_epochs}] '
                f'G: {mg:+.4f} | D: {md:+.4f} | '
                f'GP: {mgp:.4f} | Sim: {msim:.4f} | '
                f'W-dist: {w_dist:.4f} '
                f'| patience: {patience_ctr}/{cfg.patience}'
            )

        # ── Early stopping ────────────────────────────────────────────────
        if patience_ctr >= cfg.patience:
            print(
                f'Early stopping at epoch {epoch+1} — '
                f'no W-dist improvement for {cfg.patience} epochs.'
            )
            break

    # ── Restore best checkpoint ────────────────────────────────────────────
    print(f'Restoring best checkpoint (W-dist={best_w_dist:.4f})')
    G.load_state_dict(best_state_G)
    D.load_state_dict(best_state_D)

    return history


print('Starting R-GAN training...')
t0 = time.time()

history = train_rgan(G, D, fraud_loader, cfg, DEVICE)

train_time_min = (time.time() - t0) / 60
print(f'Training complete in {train_time_min:.1f} min')
wandb.log({'train/total_time_min': train_time_min})

Starting R-GAN training...


wandb: WARNING Tried to log to step 1 that is less than the current step 2. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


Epoch [ 10/300] G: -2.5170 | D: +1.1629 | GP: 0.1168 | Sim: 0.2722 | W-dist: -1.1629 | patience: 9/25
Epoch [ 20/300] G: -2.4565 | D: +1.0852 | GP: 0.0682 | Sim: 0.1024 | W-dist: -1.0852 | patience: 19/25
Early stopping at epoch 26 — no W-dist improvement for 25 epochs.
Restoring best checkpoint (W-dist=-4.3171)
Training complete in 4.4 min


In [ ]:
def generate_synthetic_fraud(G, cfg, n_samples, device, scaler):
    """Generate synthetic fraud: GAN produces GAN_FEATURES (V1-V28 + Amount)."""
    G.eval()
    all_samples = []
    with torch.no_grad():
        remaining = n_samples
        while remaining > 0:
            batch = min(remaining, 1024)
            z = torch.randn(batch, cfg.latent_dim, device=device)
            all_samples.append(G(z).cpu().numpy())
            remaining -= batch
    arr = np.vstack(all_samples)[:n_samples]
    arr = scaler.inverse_transform(arr)
    df = pd.DataFrame(arr, columns=GAN_FEATURES)
    df['Class'] = 1
    return df[GAN_FEATURES + ['Class']]


print(f'Generating {cfg.n_synthetic_fraud} synthetic fraud samples...')
synthetic_fraud = generate_synthetic_fraud(
    G, cfg, cfg.n_synthetic_fraud, DEVICE, rgan_scaler
)

print(f'Synthetic fraud shape : {synthetic_fraud.shape}')
print(f'Synthetic range       : '
      f'{synthetic_fraud[GAN_FEATURES].min().min():.1f} to '
      f'{synthetic_fraud[GAN_FEATURES].max().max():.1f}')

syn_stats = synthetic_fraud[GAN_FEATURES].describe()
wandb.log({
    'synthetic/n_samples'  : len(synthetic_fraud),
    'synthetic/amount_mean': float(syn_stats.loc['mean', 'Amount']),
    'synthetic/amount_std' : float(syn_stats.loc['std',  'Amount']),
})

In [ ]:
# Quality gates are INFORMATIONAL ONLY — paper Section V.B shows all three
# Pipeline A GANs fail JSD and discriminator AUC gates yet achieve AUROC >= 0.9998.
# Gate compliance does not predict downstream classifier utility.
def _compute_mmd(real, synthetic, gamma=1.0):
    cap = 500
    rng = np.random.default_rng(GLOBAL_SEED)
    r   = real[rng.choice(len(real),           min(cap, len(real)),      replace=False)]
    s   = synthetic[rng.choice(len(synthetic), min(cap, len(synthetic)), replace=False)]
    def rbf(A, B):
        return np.exp(-gamma * np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)).mean()
    return float(rbf(r, r) - 2 * rbf(r, s) + rbf(s, s))

def _compute_jsd(real, synthetic, n_bins=50):
    scores = []
    for i in range(real.shape[1]):
        combined = np.concatenate([real[:, i], synthetic[:, i]])
        bins     = np.linspace(combined.min(), combined.max(), n_bins + 1)
        r_prob   = np.histogram(real[:, i],      bins=bins, density=True)[0]
        s_prob   = np.histogram(synthetic[:, i], bins=bins, density=True)[0]
        r_prob  /= r_prob.sum() + 1e-12
        s_prob  /= s_prob.sum() + 1e-12
        scores.append(float(jensenshannon(r_prob, s_prob)))
    return float(np.mean(scores))

def _novelty_check(real, synthetic, min_dist=0.01, cap=200):
    rng   = np.random.default_rng(GLOBAL_SEED)
    r     = real[rng.choice(len(real),           min(cap, len(real)),      replace=False)]
    s     = synthetic[rng.choice(len(synthetic), min(cap, len(synthetic)), replace=False)]
    dists = np.sqrt(np.sum((s[:, None, :] - r[None, :, :]) ** 2, axis=-1))
    return float(dists.mean()) >= min_dist, float(dists.mean())

def _discriminability_check(real, synthetic, auc_threshold=0.75):
    X    = np.vstack([real, synthetic])
    y    = np.array([0] * len(real) + [1] * len(synthetic))
    X_sc = StandardScaler().fit_transform(X)
    clf  = LogisticRegression(max_iter=500, random_state=GLOBAL_SEED, class_weight='balanced')
    clf.fit(X_sc, y)
    auc  = float(roc_auc_score(y, clf.predict_proba(X_sc)[:, 1]))
    return auc <= auc_threshold, auc

# Use training fraud records (not holdout) for quality gate evaluation
real_arr = fraud_train_gan[GAN_FEATURES].values.astype(np.float64)
syn_arr  = synthetic_fraud[GAN_FEATURES].values.astype(np.float64)

mmd             = _compute_mmd(real_arr, syn_arr)
jsd             = _compute_jsd(real_arr, syn_arr)
nov_pass, nov   = _novelty_check(real_arr, syn_arr)
disc_pass, dauc = _discriminability_check(real_arr, syn_arr)

mmd_pass  = mmd  <= cfg.mmd_threshold
jsd_pass  = jsd  <= cfg.jsd_threshold
overall   = all([mmd_pass, jsd_pass, nov_pass, disc_pass])

print(
    f'[Synthetic Quality Gates -- INFORMATIONAL ONLY]\n'
    f'  MMD          : {mmd:.6f}  | Threshold <= {cfg.mmd_threshold}  | {"PASS" if mmd_pass  else "FAIL"}\n'
    f'  JSD (mean)   : {jsd:.6f}  | Threshold <= {cfg.jsd_threshold}  | {"PASS" if jsd_pass  else "FAIL"}\n'
    f'  Novelty dist : {nov:.6f}  | Threshold >= {cfg.novelty_min_distance}  | {"PASS" if nov_pass  else "FAIL"}\n'
    f'  Disc. AUC    : {dauc:.4f}  | Threshold <= {cfg.disc_auc_threshold}  | {"PASS" if disc_pass else "FAIL"}\n'
    f'  Overall      : {"PASS" if overall else "FAIL (informational — pipeline continues regardless)"}'
)

wandb.log({
    'quality_gates/mmd'               : mmd,
    'quality_gates/jsd_mean'          : jsd,
    'quality_gates/novelty_min_dist'  : nov,
    'quality_gates/discriminator_auc' : dauc,
    'quality_gates/overall_pass'      : int(overall),
})

In [ ]:
import lightgbm as lgb

augmented_train = pd.concat(
    [a_train[MODEL_FEATURES + ['Class']], synthetic_fraud[GAN_FEATURES + ['Class']]],
    ignore_index=True
).sample(frac=1, random_state=GLOBAL_SEED).reset_index(drop=True)

X_train_lgb = augmented_train[MODEL_FEATURES]
y_train_lgb = augmented_train['Class']
X_val_lgb   = a_val[MODEL_FEATURES]
y_val_lgb   = a_val['Class']

aug_fraud_rate = y_train_lgb.mean()
print(f'Augmented train: {augmented_train.shape} | Fraud rate: {aug_fraud_rate:.4%}')

wandb.log({
    'lgb/augmented_train_size' : len(augmented_train),
    'lgb/augmented_fraud_rate' : aug_fraud_rate,
})

In [ ]:
lgb_params = {
    'objective'         : 'binary',
    'metric'            : ['average_precision'],   # single metric avoids early stopping at round 1
    'boosting_type'     : 'gbdt',
    'num_leaves'        : cfg.lgb_num_leaves,
    'max_depth'         : -1,
    'learning_rate'     : cfg.lgb_learning_rate,
    'n_estimators'      : cfg.lgb_n_estimators,
    'min_child_samples' : cfg.lgb_min_child_samples,
    'subsample'         : cfg.lgb_subsample,
    'colsample_bytree'  : cfg.lgb_colsample_bytree,
    'reg_alpha'         : cfg.lgb_reg_alpha,
    'reg_lambda'        : cfg.lgb_reg_lambda,
    'is_unbalance'      : True,
    'random_state'      : GLOBAL_SEED,
    'n_jobs'            : -1,
    'verbose'           : -1,
}

class WandbLGBCallback:
    def __call__(self, env):
        log = {}
        for data_name, metric_name, value, _ in env.evaluation_result_list:
            log[f'lgb/{data_name}_{metric_name}'] = value
        log['lgb/boosting_round'] = env.iteration
        wandb.log(log)

dtrain = lgb.Dataset(X_train_lgb, label=y_train_lgb)
dval   = lgb.Dataset(X_val_lgb,   label=y_val_lgb, reference=dtrain)

print('Training LightGBM...')
t0 = time.time()

lgb_model = lgb.train(
    lgb_params, dtrain,
    num_boost_round = cfg.lgb_n_estimators,
    valid_sets      = [dtrain, dval],
    valid_names     = ['train', 'val'],
    callbacks       = [
        lgb.early_stopping(stopping_rounds=cfg.lgb_early_stopping_rounds, verbose=True),
        lgb.log_evaluation(period=50),
        WandbLGBCallback(),
    ]
)

lgb_time = time.time() - t0
print(f'LightGBM complete in {lgb_time:.1f}s | Best iteration: {lgb_model.best_iteration}')

wandb.log({
    'lgb/best_iteration'   : lgb_model.best_iteration,
    'lgb/training_time_sec': lgb_time,
})

fi = pd.DataFrame({
    'feature'   : lgb_model.feature_name(),
    'importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
wandb.log({'lgb/feature_importance': wandb.Table(dataframe=fi.head(20))})

In [ ]:
# ── Threshold selection on a_val (validation set only) ────────────────────
val_probs       = lgb_model.predict(X_val_lgb, num_iteration=lgb_model.best_iteration)
best_thresh, best_f1 = 0.5, 0.0
thresh_rows = []

for thresh in np.arange(0.01, 0.99, 0.005):
    preds = (val_probs >= thresh).astype(int)
    f1_t  = f1_score(y_val_lgb, preds, zero_division=0)
    p     = precision_score(y_val_lgb, preds, zero_division=0)
    r     = recall_score(y_val_lgb,    preds, zero_division=0)
    thresh_rows.append({'threshold': round(thresh, 4), 'f1': f1_t, 'precision': p, 'recall': r})
    if f1_t > best_f1:
        best_f1, best_thresh = f1_t, thresh

wandb.log({'eval/threshold_sweep': wandb.Table(dataframe=pd.DataFrame(thresh_rows))})
print(f'Best threshold (a_val): {best_thresh:.4f}  (val F1={best_f1:.4f})')

# ── Holdout evaluation at natural class distribution ──────────────────────
# Holdout accessed exactly once — natural fraud rate (~0.13%), no artificial mixing.
X_holdout_eval = holdout[MODEL_FEATURES]
y_holdout_eval = holdout['Class']

print(f'\nHoldout set: {len(holdout):,} records | '
      f'Fraud: {y_holdout_eval.sum()} ({y_holdout_eval.mean():.4%}) | '
      f'Normal: {(y_holdout_eval == 0).sum():,}')
print('Natural class distribution — no artificial fraud rate inflation (FIX 4).')

t_infer           = time.time()
holdout_probs     = lgb_model.predict(X_holdout_eval, num_iteration=lgb_model.best_iteration)
inference_time_ms = (time.time() - t_infer) * 1000
holdout_preds     = (holdout_probs >= best_thresh).astype(int)

f1        = f1_score(y_holdout_eval,        holdout_preds, zero_division=0)
precision = precision_score(y_holdout_eval, holdout_preds, zero_division=0)
recall    = recall_score(y_holdout_eval,    holdout_preds, zero_division=0)
auroc     = roc_auc_score(y_holdout_eval,   holdout_probs)
auprc     = average_precision_score(y_holdout_eval, holdout_probs)
mcc       = matthews_corrcoef(y_holdout_eval, holdout_preds)
gmean     = geometric_mean_score(y_holdout_eval, holdout_preds, zero_division=0)

print('\n' + '=' * 55)
print(' R-GAN + LightGBM — Holdout Evaluation Results')
print('=' * 55)
print(f' F1-Score  : {f1:.4f}')
print(f' Precision : {precision:.4f}')
print(f' Recall    : {recall:.4f}')
print(f' AUROC     : {auroc:.4f}')
print(f' AUPRC     : {auprc:.4f}')
print(f' MCC       : {mcc:.4f}')
print(f' G-mean    : {gmean:.4f}')
print(f' Inference : {inference_time_ms:.2f} ms ({len(X_holdout_eval):,} records)')
print(f' Threshold : {best_thresh:.4f}')
print('=' * 55)
print(classification_report(y_holdout_eval, holdout_preds, target_names=['Legit', 'Fraud']))

wandb.log({
    'holdout/f1'               : f1,
    'holdout/precision'        : precision,
    'holdout/recall'           : recall,
    'holdout/auroc'            : auroc,
    'holdout/auprc'            : auprc,
    'holdout/mcc'              : mcc,
    'holdout/gmean'            : gmean,
    'holdout/inference_time_ms': inference_time_ms,
    'holdout/threshold'        : best_thresh,
    'holdout/n_records'        : len(X_holdout_eval),
    'holdout/fraud_rate'       : float(y_holdout_eval.mean()),
})
wandb.run.summary.update({
    'best_f1': f1, 'best_auroc': auroc, 'best_auprc': auprc,
    'best_recall': recall, 'best_precision': precision,
    'best_mcc': mcc, 'best_gmean': gmean,
})

In [ ]:
results = {
    'model'               : 'R-GAN + LightGBM',
    'pipeline'            : 'A',
    'author'              : 'Bellah Ellam (bellam)',
    'f1'                  : round(f1, 4),
    'precision'           : round(precision, 4),
    'recall'              : round(recall, 4),
    'auroc'               : round(auroc, 4),
    'auprc'               : round(auprc, 4),
    'mcc'                 : round(mcc, 4),
    'gmean'               : round(gmean, 4),
    'inference_time_ms'   : round(inference_time_ms, 2),
    'threshold'           : round(best_thresh, 4),
    'n_synthetic_fraud'   : cfg.n_synthetic_fraud,
    'quality_gates_passed': overall,
    'quality_gate_detail' : {
        'mmd'         : round(mmd,  6), 'mmd_pass'    : mmd_pass,
        'jsd'         : round(jsd,  6), 'jsd_pass'    : jsd_pass,
        'novelty_dist': round(nov,  6), 'novelty_pass': nov_pass,
        'disc_auc'    : round(dauc, 4), 'disc_pass'   : disc_pass,
    },
    'architecture'        : {
        'latent_dim'   : cfg.latent_dim,
        'gen_dims'     : [cfg.gen_dim_1, cfg.gen_dim_2, cfg.gen_dim_3],
        'disc_dims'    : [cfg.disc_dim_1, cfg.disc_dim_2, cfg.disc_dim_3],
        'n_features'   : cfg.n_features,
        'lambda_sim'   : cfg.lambda_sim,
        'gp_lambda'    : cfg.gp_lambda,
        'spectral_norm': True,
        'activation'   : 'CELU',
    },
    'wandb_run_url'       : wandb.run.url,
}

out_path = Path('/kaggle/working/rgan_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

wandb.save(str(out_path))
print(f'Results saved to {out_path}')
print(json.dumps(results, indent=2))

wandb.finish()
print('W&B run finished.')